# DDI Adverse-Interaction Prediction — Comprehensive Results Summary

---

## TL;DR

- **Goal:** predict whether a pair of drugs will have an *adverse* interaction, using four independent similarity signals: biology, pharmacology, chemical structure, and protein sequence.
- **Best result:** a stacked ensemble of 52 specialized models reaches **ROC-AUC ~0.88** — clearly better than either a single model or a naive "dump everything into one model" approach.
- **Recurring pattern:** compressing a similarity score into a node2vec graph embedding almost always helps... unless a richer raw representation already exists (true for structural fingerprints), in which case the raw representation wins.
- **Known issue:** some cached/merged data files are stale (they pre-date a data-cleaning fix) and should be re-run before trusting their exact numbers — see the flag in §1 and the full list in §11.

## Glossary of diagnostic measures

This document reuses the same handful of scores over and over, in different combinations. Here's what each one actually measures, grouped by what question it answers.

**"How well does this feature predict adverse vs. non-interacting?" (model performance metrics)**
- **ROC-AUC / AUROC** (Area Under the ROC Curve): how well a model ranks adverse pairs above non-interacting pairs, across every possible decision threshold at once. 0.5 = random guessing, 1.0 = perfect separation. This is the primary metric used throughout the document, and the two names are used interchangeably.
- **PR-AUC** (Precision-Recall AUC): a companion to ROC-AUC that focuses on the precision/recall trade-off instead. It's more sensitive to how a model handles the rarer or harder-to-catch class, and is reported alongside ROC-AUC as a sanity check.
- **Accuracy**: the fraction of all predictions (adverse or non-interacting) that were correct.
- **Precision**: of the pairs the model called "adverse," what fraction really were adverse.
- **Recall**: of the pairs that really were adverse, what fraction the model actually caught.
- **F1**: a single number balancing precision and recall (their harmonic mean) — useful when you care about both and want one summary score.

**"Is this similarity graph healthy enough to trust?" (network/graph diagnostics, run before node2vec)**
- **auroc_ddi_prediction**: takes the raw similarity score sitting on each graph edge (before any node2vec embedding) and checks how well *that score alone* separates real adverse-DDI edges from random drug pairs. This is the "raw signal strength" of a feature, i.e. the ceiling that node2vec embedding is then compared against.
- **isolate rate**: the percentage of drugs (nodes) that end up with zero edges after the graph is sparsified — meaning their similarity to every other drug was too weak or got pruned. A high isolate rate means a lot of drugs have no usable signal at all for that feature.
- **p5_degree**: the 5th-percentile node degree (number of edges) across the whole graph — a connectivity floor. If `p5_degree = 0`, at least 5% of nodes have little or no connectivity, which is a red flag for any graph-based method like node2vec.
- **rho_size_diff**: the Spearman correlation between an edge's weight (similarity score) and how different the two drugs' "profile sizes" are (e.g. one drug has 40 known targets, the other has 2). A strong correlation here means the similarity score is partly just measuring "how much data do we have on this drug," not genuine biological similarity — a data-leakage risk, not a real signal.
- **degree_vs_size_slope**: the slope of a regression line fitting a node's degree (number of connections) against its profile size (e.g. number of ATC codes). A high slope means well-documented drugs simply accumulate more graph connections regardless of true similarity — another flavor of the same leakage risk as `rho_size_diff`.
- **auroc_edge_vs_random**: after node2vec produces embeddings, this checks whether real graph edges have higher embedding cosine similarity than random node pairs. It only confirms the embedding faithfully reproduced the graph's structure — it does *not* measure whether the embedding is useful for predicting adverse DDIs (that's what §2.4/§3.4/§4.3's raw-vs-embedding comparisons are for).
- **mean edge cosine similarity**: the average cosine similarity between the node2vec embeddings of two connected drugs. Used to compare how "tightly" the adverse graph embeds vs. the non-interacting graph — a big gap between the two is flagged repeatedly as a possible classifier shortcut worth double-checking, since it could reflect real signal or an artifact of how the graphs were built.

**"Are two features actually different, or just restating the same thing?" (correlation / statistical tests)**
- **Pearson correlation**: measures the *linear* relationship between two similarity scores (e.g. do structural similarity and ATC similarity rise and fall together?). Ranges from -1 to 1; near 1 means the two features are largely redundant, near 0 means they carry mostly independent information — this is what drives the "convergent vs. divergent feature" calls in §6.
- **Spearman correlation**: the same idea as Pearson but based on rank order rather than raw values, making it more robust to outliers. Used in §5.1 to compare how similar the three candidate amino-acid-similarity kernels are to each other.
- **Chi-square (χ²) test**: a statistical test for whether a categorical variable's distribution (e.g. "shared ATC depth" bucket) differs between adverse and non-interacting pairs. `p` is the probability that a difference this large would show up by chance alone (smaller = more confident there's a real difference); `df` (degrees of freedom) is a technical parameter of the test based on the number of categories compared. A significant chi-square result doesn't by itself mean the effect is *large* — that's why §3.1 pairs it with the (weak) AUROC number.

**"How much does the model actually rely on this piece of the data?" (model-internal measures)**
- **Feature-group importance (RF split-importance)**: how often a Random Forest used a given block of columns to make a decision split, normalized so all groups add up to 1. Higher = the model leaned on that group more — but this can understate a wide, information-rich block (like 4,096 fingerprint bits) relative to a compact block (like 128 embedding dimensions), simply because importance gets spread thin across more columns (see §7.2).
- **Meta-learner coefficients**: in the stacked ensemble (§8), a logistic-regression "meta-learner" assigns a weight to each of the 52 experts' predictions when combining them. The size of a coefficient (`|coefficient|`) reflects how much that expert's opinion swings the final combined prediction; sign (+/-) shows whether it pushes toward "adverse" or away from it.

---

## 0. Project overview

**Goal:** predict whether a pair of drugs has an *adverse* interaction (label = 1) vs. is *non-interacting* (label = 0), using four independent, complementary similarity "hypotheses":

| Hypothesis | Feature domain | Folder |
|---|---|---|
| **H1** | Biological target/pathway overlap (9 layers: target identity ladder, GO terms, Pfam domains, pathways) | [notebooks/h1_biological_overlap/](h1_biological_overlap) |
| **H2** | Pharmacological/clinical similarity (ATC codes) | [notebooks/h2_pharmacological_similarity/](h2_pharmacological_similarity) |
| **H3** | Chemical structural similarity (Morgan/ECFP fingerprints) | [notebooks/h3_structural_similarity/](h3_structural_similarity) |
| **H4** | Target amino-acid sequence similarity | [notebooks/h4_target_amino_acid_sim/](h4_target_amino_acid_sim) |

**Headline result:** a stacked meta-learner combining 52 specialized (view, algorithm) experts reaches **ROC-AUC 0.878–0.888** (depending on whether H4 is included). That clearly beats both:
- a naive "kitchen-sink" model that concatenates every feature into one Random Forest (ROC-AUC ~0.79–0.80), and
- the single best individual expert (raw Morgan fingerprint bits + Extra Trees, ROC-AUC ~0.82–0.83).

---

## 1. Data curation — [data_curation/](data_curation)

Canonical version: [data_curation_notebook_v2.ipynb](data_curation/data_curation_notebook_v2.ipynb). Superseded: [data_curation_notebook.ipynb](data_curation/data_curation_notebook.ipynb) (v1).

**Pipeline steps:**
1. **Build the adverse dataset** — start from the full DrugBank approved-drug DDI pair list and remove pairs whose description indicates a purely beneficial interaction (e.g. "therapeutic efficacy increased").
2. **Build the negative (non-interacting) dataset** — randomly sample drug pairs with **no** documented interaction at all, class-balanced against the adverse dataset.
3. **Enrich every drug** with core identity features pulled from the DrugBank XML dump: SMILES, ATC codes, target UniProt IDs, target FASTA sequences.
4. **Backfill missing features** in two passes using PubChem, ChEMBL, UniProt, and the WHO ATC/DDD index.
5. **Filter both pair datasets** down to only pairs where *both* drugs have a complete core feature set.

**Funnel (exact printed numbers):**

| Stage | Adverse | Negative |
|---|---|---|
| Raw DrugBank DDI export | 1,129,743 pairs | — |
| Beneficial pairs removed | −15,971 | — |
| Kept (adverse-only) | 1,113,772 | 1,113,772 (1:1 sampled, seed=42, from 9,130,829 available non-documented pairs) |
| Small-molecule pair filter | 818,997 | 570,462 |
| Feature-completeness gate (`has_all_features`, after 2-pass backfill) | 57.3% of 3,316 XML-matched drugs | — |
| **Final saved (current, on-disk)** | **471,895 rows**, 1,893 unique drugs | **161,098 rows**, 1,893 unique drugs |

**Drug inclusion rule:** `type == "small molecule"` and not withdrawn — deliberately includes both approved and unapproved small molecules; the feature-completeness gate is the real filter, not approval status.

**The "symmetric gate" — v2's key fix over v1:**
v2 adds a rule requiring every surviving drug to appear in *both* the negative and adverse pair sets. Without this, a drug that only ever appears in one population would let a downstream classifier "cheat" by keying off drug identity instead of the actual similarity signal.

v1's actual final output (no symmetric gate) shows the leak this fixes:
```
Negative: 569,934 -> 162,893   (1,902 unique drugs)
Adverse : 830,623 -> 478,324   (1,900 unique drugs)
Union: 1,907 unique drugs  ->  7 drugs appear in only ONE of the two populations
```
v2 closes this gap entirely (1,893/1,893, 0 asymmetric drugs).

## 2. H1 — Biological overlap — [h1_biological_overlap/](h1_biological_overlap)

### 2.1 Raw similarity computation — [biological_overlap.ipynb](h1_biological_overlap/biological_overlap.ipynb)

9 layers, each with a `{layer}_jaccard` (symmetric) and two Tversky (directional-containment) scores:

| Layer | Data source | Ladder logic |
|---|---|---|
| `identity_L0` | DrugBank targets ∪ ChEMBL mechanism/bioactivity targets (pChEMBL ≥ 6.0) | base set |
| `identity_L1` | `L0` ∪ enzymes ∪ transporters | +2 protein roles |
| `identity_L2` | `L1` ∪ carriers | +1 more role |
| `go_mf`/`go_bp`/`go_cc` | DrugBank XML GO annotations | split by aspect |
| `pfam` | DrugBank Pfam domains | — |
| `pathway_native` | SMPDB pathway-protein map + DrugBank native pathway IDs | direct membership |
| `pathway_inferred` | Same SMPDB map, reseeded from `L0` | two-hop (L0 proteins → their pathways → all proteins in those pathways) |

Final: 1,893 unique drugs; 471,895 adverse / 161,098 non-interacting pairs.

### 2.2 Network diagnostics — [network_generation.ipynb](h1_biological_overlap/network_generation.ipynb)

Sparsification: hybrid top-k (`K_NEIGHBORS=15`, `MIN_SIMILARITY=1e-9`), shared across all 9 layers.

| Layer | Adverse AUROC (DDI vs. random) | Adverse isolate rate | Non-int isolate rate |
|---|---|---|---|
| **`pathway_inferred`** | **0.887** (strongest) | 16% | 16% |
| **`go_cc`** | **0.845** | 12% | 12% |
| `identity_L2` | 0.764 | 12% | 13% |
| `identity_L1` | 0.755 | 13% | 14% |
| `go_bp` | 0.698 | 15% | 14% |
| `go_mf` | 0.653 | 15% | 16% |
| `pfam` | 0.593 | 19% | 24% |
| `pathway_native` | 0.523 (~coin flip) | **81%** | **81%** |
| `identity_L0` | 0.531 (~coin flip) | 25% | **50%** |

All 18 graphs (9 layers × 2 populations) fail two diagnostic checks: the connectivity bar (`p5_degree ≥ 5`, but every graph scores `p5_degree = 0.0`) and the edge-weight size-leakage check (`rho_size_diff` ranges from −0.18 to **−0.90**, worst for `pfam`) — meaning edge weight is confounded with how large a drug's biological profile is. **`pathway_native` and `identity_L0` are the clearest, most severe weak points in the entire project**, effectively unusable for node2vec as currently configured.

### 2.3 node2vec embeddings — [node2vec_embedding.ipynb](h1_biological_overlap/node2vec_embedding.ipynb)

`NODE2VEC_PARAMS = dict(dimensions=64, walk_length=40, num_walks=8, p=1.0, q=1.0, window=10, epochs=5, seed=42)` — deliberately reduced vs. H2/H3/H4's `128/80/10` since 9× more graphs need embedding here.

All 18 embeddings pass the structural sanity check (`auroc_edge_vs_random` 0.964–0.999). Recurring pattern: **adverse embeds tighter than non_interacting** for every layer except the two sparsest (`identity_L0`, `pathway_native`) — e.g. `go_mf`: mean edge cosine sim 0.797 (adverse) vs. 0.540 (non_interacting); `pfam`: 0.808 vs. 0.649. Flagged project-wide as a possible classifier shortcut risk worth ablating.

### 2.4 Raw scalar vs. node2vec embedding — [biological_overlap_embedding_comparison.ipynb](h1_biological_overlap/biological_overlap_embedding_comparison.ipynb)

RF, 40k train / 10k test, identical split per layer:

| Layer | similarity_score (roc_auc) | node2vec_embedding (roc_auc) | gap |
|---|---|---|---|
| `identity_L0` | 0.525 | 0.771 | **+0.246** |
| `go_mf` | 0.559 | 0.769 | +0.210 |
| `pfam` | 0.551 | 0.755 | +0.204 |
| `go_bp` | 0.571 | 0.757 | +0.186 |
| `go_cc` | 0.592 | 0.758 | +0.166 |
| `pathway_inferred` | 0.589 | 0.736 | +0.147 |
| `pathway_native` | 0.514 | 0.645 | +0.132 |
| `identity_L1` | 0.670 | 0.770 | +0.100 |
| `identity_L2` | 0.673 | 0.764 | +0.091 |

**node2vec beats the raw Jaccard scalar in all 9/9 layers, no exceptions** — the single most consistent finding in the whole project. `pathway_native` remains weakest even after embedding (0.645), a real information ceiling from its 81% isolate rate that better features only partially recover.

---

## 3. H2 — Pharmacological (ATC) similarity — [h2_pharmacological_similarity/](h2_pharmacological_similarity)

### 3.1 Raw similarity — [atc_similiarity_notebook.ipynb](h2_pharmacological_similarity/atc_similiarity_notebook.ipynb)

Depth-weighted ATC-code matching: level 2/3/4 code-prefix matches weighted $2^{d-1}$ (weights 1/2/4, denominator 7), "Mean-of-Max" aggregation across multi-code drugs, symmetrized. A chi-square test on shared-ATC-depth categories vs. adverse/non-interacting label: **χ² = 10,655.89, df = 3, p ≈ 0** — statistically significant but (see below) practically weak.

### 3.2 Network diagnostics — [network_generation.ipynb](h2_pharmacological_similarity/network_generation.ipynb)

| Dataset | n_edges | isolate rate | degree_vs_size_slope | auroc_ddi_prediction |
|---|---|---|---|---|
| Adverse | 13,826 | 9.9% | **0.670** (high — ATC-code count strongly predicts degree) | **0.538** (barely > chance) |
| Non-interacting | 5,031 | **22.5%** | 0.338 | NaN |

**ATC similarity alone is only weakly predictive of adverse DDI status** (raw AUROC 0.538) — the weakest raw signal of any hypothesis in the project, and the `degree_vs_size_slope=0.67` flags a real (if mild) leakage risk from ATC-code-count.

### 3.3 node2vec embeddings — [node2vec_embedding.ipynb](h2_pharmacological_similarity/node2vec_embedding.ipynb)

`NODE2VEC_PARAMS`: `dimensions=128, walk_length=80, num_walks=10, p=q=1.0, window=10, epochs=5`.
`auroc_edge_vs_random` = 0.995 (adverse) / 0.983 (non-interacting) — near-perfect structural fidelity. But the **real** DDI-predictiveness check — embedding cosine similarity vs. known adverse pairs — gives **AUROC 0.646**, vs. the raw ATC score's 0.538: a modest +0.11 AUROC improvement over a weak baseline. The non-interacting graph embeds noticeably "patchier" than the adverse graph (mean edge cosine 0.71 vs. 0.88), attributed to its higher isolate rate.

### 3.4 Raw scalar vs. node2vec embedding — [atc_sim_embedding_comparison.ipynb](h2_pharmacological_similarity/atc_sim_embedding_comparison.ipynb)

| Model | dimensionality | accuracy | roc_auc | pr_auc |
|---|---|---|---|---|
| `similarity_score` | 1 | 0.523 | **0.522** | 0.519 |
| `node2vec_embeddings` | 128 | 0.717 | **0.800** | 0.806 |

**+0.278 ROC-AUC gap — the largest scalar-vs-embedding gap of any hypothesis.** The raw scalar predicts "non-interacting" almost every time (adverse-class recall only 0.075); node2vec is likely close to the practical ceiling for this feature type since ATC codes have no fixed-size raw representation to compare against (unlike H3's fingerprints).

---

## 4. H3 — Structural (Morgan fingerprint) similarity — [h3_structural_similarity/](h3_structural_similarity)

### 4.1 Network diagnostics — [network_generation.ipynb](h3_structural_similarity/network_generation.ipynb)

| Dataset | n_edges | isolates | auroc_ddi_prediction |
|---|---|---|---|
| Adverse | 20,776 | 6 (0.3%) | **0.990** (strongest raw signal in the whole project) |
| Non-interacting | 20,295 | 2 (0.1%) | NaN |

By far the best-connected graph type of any hypothesis (isolate rate 0.1–0.3% vs. H1's 12–81%, H2's 9.9–22.5%).

### 4.2 node2vec embeddings — [node2vec_embedding.ipynb](h3_structural_similarity/node2vec_embedding.ipynb)

`NODE2VEC_PARAMS`: `dimensions=128, walk_length=80, num_walks=10, p=q=1.0`. `auroc_edge_vs_random` = 0.987 (adverse) / 0.995 (non-interacting). Unlike H2, both populations embed comparably well here (no patchiness asymmetry).

### 4.3 Three-way model comparison — [structural_sim_embedding_comparison.ipynb](h3_structural_similarity/structural_sim_embedding_comparison.ipynb)

RF, 40k train / 10k test, identical split for all three:

| Model | dimensionality | accuracy | roc_auc | pr_auc |
|---|---|---|---|---|
| `similarity_score` (raw Tanimoto) | 1 | 0.531 | 0.554 | 0.550 |
| `node2vec_embeddings` | 128 | 0.692 | 0.760 | 0.752 |
| **`fingerprint_bits`** (raw Morgan, no graph) | 4096 | **0.760** | **0.837** | **0.826** |

**Raw fingerprint bits win outright, node2vec is a clear second, raw scalar trails far behind.** This is the only hypothesis with a fixed-size "raw representation" alternative to graph compression, and it beats the embedding by +0.08 ROC-AUC — node2vec is fundamentally capped by how much information survives the graph-sparsification step (top-15-neighbor + min-similarity), it can never "see" the original fingerprint bits directly.

### 4.4 Standalone classifier — [simple_binary_classifcation_model.ipynb](h3_structural_similarity/simple_binary_classifcation_model.ipynb)

Fingerprint-only models (25k/class, 40k train/10k test): `random_forest` roc_auc **0.8377** (best), `logistic_regression` 0.8363. Separately, a similarity-score-only logistic regression on the full imbalanced dataset (513k train / 128k test) scores only **roc_auc 0.576** — consistent with §4.3's finding that the raw scalar is weak in isolation. *(This notebook has no written conclusions section — a documentation gap.)*

---

## 5. H4 — Target amino-acid similarity — [h4_target_amino_acid_sim/](h4_target_amino_acid_sim)

### 5.1 Kernel comparison — [amino_acid_sim_notebook.ipynb](h4_target_amino_acid_sim/amino_acid_sim_notebook.ipynb)

Hypothesis: "Adverse drug pairs act on proteins that are more evolutionarily/structurally similar (by amino-acid sequence) than non-interacting pairs, even when the exact targets differ."

Three candidate kernels compared on an 800/800 sample:

| Kernel | Method | Best metric AUROC vs. label |
|---|---|---|
| **`kmer_cosine`** (winner) | Tripeptide-composition cosine similarity (`CountVectorizer`, 3-mers, 8,048-term vocab) | **0.614** (`max_similarity`) |
| `kmer_fp` | Hashed 3-mer fingerprint + Tanimoto (2048 bits) | 0.586 |
| `alignment` | Needleman-Wunsch + BLOSUM62, sampled subset only (too slow at full scale) | 0.557, p=0.13 for `mean_similarity` (not significant) |

`kmer_cosine`'s `best_match_avg` aggregation was selected as the production `similarity_score`. Explicitly documented as **"a first-pass hypothesis check on a comparison sample, not a validated production kernel."** Inter-kernel Spearman: `kmer_fp` vs `kmer_cosine` = 0.886 (largely redundant); both vs. `alignment` ≈ 0.23–0.27 (largely independent — alignment-based similarity may carry real complementary signal not yet exploited).

Full-scale final counts: **471,169 adverse / 162,788 non-interacting pairs.**

### 5.2 Network diagnostics — [network_generation.ipynb](h4_target_amino_acid_sim/network_generation.ipynb)

| Dataset | n_edges | isolates | auroc_ddi_prediction |
|---|---|---|---|
| Adverse | 20,868 | 106 (5.6%) | **0.970** (2nd-strongest raw signal in the project, after H3's 0.990) |
| Non-interacting | 23,315 | 105 (5.5%) | NaN |

### 5.3 node2vec embeddings — [node2vec_embedding.ipynb](h4_target_amino_acid_sim/node2vec_embedding.ipynb)

Same paper-default params as H2/H3 (`dim=128, walk=80, walks=10, p=q=1`). `auroc_edge_vs_random` = 0.963 (adverse) / 0.979 (non-interacting) — structurally sound. **No `amino_acid_sim_embedding_comparison.ipynb` exists yet** — H4 has never had its raw-vs-embedding DDI-predictiveness gap formally measured the way H1/H2/H3 have. This session's `ensemble_model.ipynb` run is the first data point: `h4_amino_acid`'s best expert (`extra_trees`) only reaches **ROC-AUC 0.749** — a large, ~0.22-point drop from its 0.970 raw signal, comparable to H3's embedding-compression loss (0.990→0.741). Since H4 (unlike H3) has no raw-fingerprint-style expert yet, this is flagged as the single highest-value fix identified this session (§8, §10).

---

## 6. Convergence/divergence analysis — [convergance_divergence_analysis.ipynb](convergance_divergence_analysis.ipynb)

Analyzes all 11 pre-H4 similarity layers (H1's 9 + H2's ATC + H3's structural; **H4 is not yet included in this notebook**).

### 6.1 Structural-vs-ATC pairwise quadrant breakdown

| Category | Adverse (n=478,324†) | Non-interacting (n=162,893†) |
|---|---|---|
| Concordant (low struct / low ATC) | 96.5% | 99.0% |
| Concordant (high struct / high ATC) | 0.03% | 0.006% |
| **Divergent — scaffold hop** (low struct / high ATC) | **3.5%** | **1.0%** |
| Divergent — repurposed (high struct / low ATC) | 0.006% | 0.002% |

†Note: this notebook's row counts (478,324/162,893) reflect the same stale, pre-symmetric-gate snapshot flagged in §1 — re-running against current v2 data is recommended, though the *proportions* are unlikely to shift much.

**Scaffold-hopping divergence is 3.5× more common in adverse pairs.** This suggests adverse interactions are often driven by a shared pharmacodynamic mechanism or receptor class rather than shared chemistry. Pearson(structural, ATC) = 0.29 (adverse) / 0.13 (non-interacting) — the two lenses are largely independent, justifying combining both as complementary features.

### 6.2 Full 11×11 correlation matrix (Pearson, adverse pairs)

| | L0 | L1 | L2 | GO-MF | GO-BP | GO-CC | Pfam | Path-nat | Path-inf | ATC | Struct |
|---|---|---|---|---|---|---|---|---|---|---|---|
| **L0** | 1.00 | 0.54 | 0.53 | 0.83 | 0.87 | 0.68 | 0.43 | 0.27 | 0.59 | 0.35 | 0.27 |
| **L1** | 0.54 | 1.00 | **0.98** | 0.46 | 0.46 | 0.38 | 0.21 | 0.15 | 0.29 | 0.22 | 0.22 |
| **L2** | 0.53 | **0.98** | 1.00 | 0.46 | 0.45 | 0.37 | 0.20 | 0.15 | 0.29 | 0.21 | 0.22 |
| **GO-MF** | 0.83 | 0.46 | 0.46 | 1.00 | 0.86 | 0.71 | 0.46 | 0.26 | 0.61 | 0.36 | 0.27 |
| **GO-BP** | 0.87 | 0.46 | 0.45 | 0.86 | 1.00 | 0.73 | 0.46 | 0.26 | 0.63 | 0.37 | 0.28 |
| **GO-CC** | 0.68 | 0.38 | 0.37 | 0.71 | 0.73 | 1.00 | 0.46 | 0.20 | 0.63 | 0.28 | 0.24 |
| **Pfam** | 0.43 | 0.21 | 0.20 | 0.46 | 0.46 | 0.46 | 1.00 | 0.13 | 0.63 | 0.18 | 0.21 |
| **Path-nat** | 0.27 | 0.15 | 0.15 | 0.26 | 0.26 | 0.20 | 0.13 | 1.00 | 0.20 | 0.16 | 0.14 |
| **Path-inf** | 0.59 | 0.29 | 0.29 | 0.61 | 0.63 | 0.63 | 0.63 | 0.20 | 1.00 | 0.22 | 0.25 |
| **ATC** | 0.35 | 0.22 | 0.21 | 0.36 | 0.37 | 0.28 | 0.18 | 0.16 | 0.22 | 1.00 | 0.29 |
| **Struct** | 0.27 | 0.22 | 0.22 | 0.27 | 0.28 | 0.24 | 0.21 | 0.14 | 0.25 | 0.29 | 1.00 |

(Non_interacting matrix shows the same clusters, generally weaker: e.g. L1/L2 = 0.97, L0/GO-BP = 0.81.)

**Convergent (redundant) cluster:** `identity_L1`/`identity_L2` almost collinear (0.98/0.97) — L2 only adds carrier proteins on top of L1. `identity_L0` also converges strongly with `go_bp`/`go_mf` (0.87/0.83) since shared targets mechanically drive shared GO annotations.

**Divergent (complementary) features:** `atc_sim`, `structural_sim`, and especially `pathway_native` (0.13–0.29 Pearson with everything else) stay weakly correlated with the biological block and each other — "the strongest candidates for adding genuinely non-redundant axes."

**Practical implication:** consider collapsing the identity ladder to L0 + L2 only, or dropping one GO aspect. `atc_sim`, `structural_sim`, and `pathway_native` remain comparatively decorrelated from everything else, and are the best candidates for adding genuinely new signal.

No PCA/clustering/dendrogram analysis exists in this notebook yet — an open follow-up.

---

## 7. Feature merging & combined RF baseline

### 7.1 [merging_into_final_drug_pair_feature_table.ipynb](merging_into_final_drug_pair_feature_table.ipynb)

Merges H1 + H2 + H3 (⚠️ **not H4**) on `(drug1_id, drug2_id)` with duplicate-occurrence-rank matching and `validate="one_to_one"`. Cached output (stale, see §1): 478,324×31 adverse / 162,893×31 non-interacting unified table, 0 NaNs introduced. No conclusions/recommendations markdown exists in this notebook.

### 7.2 Combined RF baseline — `models/combined_rf_model/`

```
accuracy=0.7295, precision=0.7136, recall=0.7666, f1=0.7392, roc_auc=0.8093, pr_auc=0.8197
```

Feature-group importance ranking (normalized RF split-importance):

| Rank | Group | n_cols | Importance |
|---|---|---|---|
| 1 | `h2_atc` | 128 | 0.170 |
| 2 | `h1_identity_L0` | 64 | 0.105 |
| 3 | `h1_identity_L1` | 64 | 0.095 |
| 4 | `h1_identity_L2` | 64 | 0.091 |
| 5 | `h1_go_mf` | 64 | 0.082 |
| 6 | `h1_pfam` | 64 | 0.081 |
| 7 | `h1_pathway_native` | 64 | 0.078 |
| 8 | `h1_go_cc` | 64 | 0.077 |
| 9 | `h1_go_bp` | 64 | 0.077 |
| 10 | `h1_pathway_inferred` | 64 | 0.075 |
| 11 | `h3_fingerprint` | **4,096** | **0.070** (lowest, despite most columns by far) |

**Important cross-check against §8 (ensemble model):** here, the 4,096-dim `h3_fingerprint` block ranks *lowest* in aggregate importance despite being the single strongest expert view once modeled in isolation (§8). This is not a contradiction — it's exactly the "kitchen-sink dilution" effect §8 documents directly: a single concatenated RF spreads split-importance thin across thousands of individual bit-columns, systematically undervaluing a wide, information-rich block relative to a handful of compact, already-compressed embedding blocks like `h2_atc` (128 cols). ⚠️ Also recall this model was trained on the stale, pre-symmetric-gate merged table (§1) and has no reproducible source notebook — treat its absolute numbers as provisional pending a re-run.

---

## 8. Ensemble model — [models/ensemble_model.ipynb](models/ensemble_model.ipynb)

Trains 4 algorithm families (Random Forest, Extra Trees, Decision Tree, Logistic Regression) × 13 feature views (9 H1 layers + H2 ATC + H3 structural + H3 fingerprint + H4 amino-acid) = **52 specialized experts**, all on an identical shared 24k-train/6k-test split (`MAX_PAIRS_PER_CLASS=15,000`), then combines them via soft voting, ROC-AUC-weighted voting, and a stacked out-of-fold logistic-regression meta-learner. Also benchmarks a naive "kitchen-sink" RF (all 13 views concatenated, 5,056 raw dims).

### 8.1 Headline comparison (run without H4, 48 experts, vs. with H4, 52 experts)

| Model | ROC-AUC (12 views) | ROC-AUC (13 views, +H4) |
|---|---|---|
| Best single expert (`h3_fingerprint__extra_trees`) | 0.827 | 0.824 |
| Kitchen-sink RF (all views concatenated) | 0.801 | 0.791 |
| Soft voting (unweighted) | 0.822 | 0.812 |
| ROC-AUC-weighted voting | 0.828 | 0.818 |
| **Stacked meta-learner** | **0.888** | **0.878** |

### 8.2 Individual expert ranking (top/bottom, 13-view run)

| Expert | ROC-AUC |
|---|---|
| `h3_fingerprint__extra_trees` (best overall) | 0.824 |
| `h2_atc__extra_trees` | 0.781 |
| `h1_identity_L1__extra_trees` (best H1 layer) | 0.764 |
| `h4_amino_acid__extra_trees` | 0.749 |
| ⋯ (mid-pack H1 layers) | 0.72–0.75 |
| `h1_pathway_native__logistic_regression` (worst overall) | 0.520 |

Algorithm ranking holds on **every** view without exception: `extra_trees > random_forest > logistic_regression > decision_tree`. `decision_tree` is frequently barely better than chance and contributes least to the meta-learner (mean |coefficient| ≈ 0.17–0.18, lowest of any family in both runs).

### 8.3 Key findings

1. **Kitchen-sink concatenation is beaten by every combination rule, and even by the single best expert alone** in both runs — naive concatenation dilutes signal rather than combining it (see §7.2's cross-check for why).
2. **The stacked meta-learner is the clear winner**, beating both voting schemes by 0.05–0.07 ROC-AUC — it can learn non-uniform, even negative, per-expert weights that simple averaging cannot.
3. **Adding H4 as a 13th view slightly *lowered* every combination metric** (~0.01 ROC-AUC across the board), despite H4's raw signal (0.970 AUROC) being the second-strongest in the project. Root cause: H4's node2vec embedding compresses that raw signal down to ROC-AUC 0.749 (a ~0.22-point loss), and — unlike H3 — H4 has no matching raw-representation expert to recover that loss. The meta-learner still assigns H4's experts real, positive weight (`logistic_regression` +0.586, `random_forest` +0.538, `extra_trees` +0.510), so H4 isn't valueless; the net dip is most likely a mix of genuine dilution and a slightly different sampled train/test pool (H4 coverage drops ~1,300 pairs from the shared intersection under the same `RANDOM_STATE=42`).

Full per-run artifacts: `models/ensemble_rf_dt_lr/expert_results.csv`, `ensemble_results.csv`, `meta_learner_coefficients.csv`.

---

## 9. Cross-cutting conclusions

1. **node2vec embeddings beat their raw similarity scalar in every hypothesis without exception** (H1: all 9/9 layers; H2: +0.278 ROC-AUC; H4: not yet formally measured but implied). The one exception to "richer feature = better" is H3, where the *raw fingerprint bits* (not a scalar, a 4096-dim fixed-size vector) beat node2vec by +0.08 — node2vec is a lossy compression, and it only wins when no richer fixed-size raw representation exists.
2. **H3 (structural) and H4 (amino-acid) have by far the strongest raw, pre-compression signals** in the project (raw AUROC 0.990 and 0.970 respectively) — far stronger than H1's best layer (`pathway_inferred`, 0.887) or H2's ATC (0.538). Both lose a large chunk of that signal (~0.22–0.25 ROC-AUC) once compressed into a 128-dim node2vec embedding.
3. **H1's `pathway_native` and `identity_L0` are the clearest, most consistently-flagged weak points** across every notebook that touches them (81%/25–50% isolate rates, raw AUROC ~0.52, weakest post-embedding AUROC of any layer at 0.645).
4. **`identity_L1`/`identity_L2` are redundant** (Pearson 0.98) and **`atc_sim`/`structural_sim`/`pathway_native` are the most complementary (decorrelated) features** — directly actionable for feature selection.
5. **Specialized per-view modeling + a learned combiner beats naive concatenation everywhere it's been tested** — both in the ensemble model (§8) and implicitly in the combined-RF baseline's feature-importance dilution (§7.2).
6. **"Adverse embeds tighter than non-interacting"** is a recurring pattern across H1 (most layers), H2, and (implicitly, via isolate-rate asymmetry) H4 — flagged repeatedly as a possible classifier shortcut worth ablating, never yet formally tested.

---

## 10. Concrete strategy going forward

(See also [RESULTS_09_4_26.md](RESULTS_09_4_26.md) for the version of this list written immediately after the H4 ensemble run.)

1. **Give H4 a raw-representation expert**, analogous to H3's fingerprint-bits expert — a per-drug k-mer composition vector (already computed as `kmer_cosine`/`kmer_fp` in `amino_acid_sim_notebook.ipynb`), max/min-combined per pair. Highest expected-value single fix identified this session.
2. **Re-tune or drop `pathway_native`/`identity_L0`'s sparsification** (H1) — they need their own `K_NEIGHBORS`/`MIN_SIMILARITY` via `network_builder.py`'s diagnostic sweep, not the shared setting.
3. **Drop the near-duplicate `identity_L1`/`identity_L2` pair** (§6.2) — keep one, freeing an expert slot for genuinely complementary signal.
4. **Retire `decision_tree` as an algorithm family**; replace with `HistGradientBoostingClassifier` — it is dominated by `random_forest`/`extra_trees` on every single view tested.
5. **Re-run `merging_into_final_drug_pair_feature_table.ipynb` and a combined-RF notebook against the current v2 data** (§1, §7) — the existing artifacts are stale and their absolute numbers shouldn't be quoted without re-validation. Also add H4 to this merge (currently H1+H2+H3 only).
6. **Explore biased random walks (p≠1, q≠1)** — every graph in the project currently uses unbiased p=q=1 walks; H2's ATC and H4's amino-acid graphs in particular have strong-to-decent raw signal that their embeddings underperform.
7. **Calibrate expert probabilities before combining** (`CalibratedClassifierCV`) and **L1-regularize the meta-learner** to explicitly zero out low-value experts rather than only shrinking their coefficients.
8. **Build `amino_acid_sim_embedding_comparison.ipynb`** (H4's missing three-way comparison notebook) to formally quantify its raw-vs-embedding gap the way H1/H2/H3 already have.
9. **Ablate the "adverse embeds tighter" pattern** (finding #6, §9) via feature importance / permutation tests to rule out a classifier shortcut before trusting absolute performance numbers.
10. **Add a PCA/clustering pass to `convergance_divergence_analysis.ipynb`** and extend it to include H4 — currently absent.
11. **Re-run with `MAX_PAIRS_PER_CLASS=None`** (all available pairs) once the above is locked in, to confirm conclusions hold at full scale.

---

## 11. Known documentation drift / integrity issues (consolidated)

| # | Issue | Location |
|---|---|---|
| 1 | Merged feature table + combined RF model trained on stale, pre-symmetric-gate data; no reproducible source notebook for the combined RF artifacts | §1, §7 |
| 2 | `network_generation.ipynb` (H2) markdown quotes stale isolate/edge counts (423/1,900, 22.3%) vs. actual (426/1,893, 22.5%) | H2 §3.2 |
| 3 | `atc_sim_embedding_comparison.ipynb` conclusion table is a slightly earlier re-run than the current executed `results_df` (same conclusions, 3rd-decimal drift only) | H2 §3.4 |
| 4 | `simple_binary_classifcation_model.ipynb` (H3) has no written conclusions section despite training 3 models on 2 datasets | H3 §4.4 |
| 5 | H1's `network_generation.ipynb`/`node2vec_embedding.ipynb` markdown cites slightly stale isolate/edge counts from an earlier 1,894-drug run (current: 1,893) — cosmetic, doesn't change conclusions | H1 §2.2–2.3 |
| 6 | H4 has no `amino_acid_sim_embedding_comparison.ipynb` yet — its raw-vs-embedding gap has only been indirectly observed via this session's ensemble run | H4 §5.3 |

---

## 12. Legacy / exploratory notebooks — [notebooks_test/](../notebooks_test)

Early exploratory work (DrugBank XML parsing, ChEMBL/UniChem ID mapping, ATC-similarity prototyping, LLM-based classification exploration) that directly preceded and informed the current `data_curation` → H1–H4 pipeline. Superseded, not part of the current results pipeline, kept for reference:

- `drugbank raw data extraction.ipynb` (+ archived variant) — DrugBank XML parsing prototype
- `db_to_chembl_through_unichem.ipynb` — DrugBank↔ChEMBL ID mapping
- `ChEMBL Target Profiles - Exploratory.ipynb` / `ChEMBL Taregt Profile - Test Case.ipynb` — ChEMBL target-profile pulls
- `protein_target_data_collection.ipynb` — protein target collection (precursor to H1/H4 target features)
- `ATC Similarity Demo.ipynb` — prototype of H2's ATC hierarchical-similarity scheme
- `general_network_construction.ipynb` — early network-building prototype
- `negative_ddi_sampling.ipynb` / `non_adverse_ddi_extraction.ipynb` / `drug_interaction_desc_search.ipynb` — pair-curation prototypes (precursors to `data_curation_notebook_v2.ipynb`)
- `llm_classification_analysis.ipynb` — analysis of the separate LLM-based DDI category classification runs (`data/raw/lmm_classifcation_runs/`)
- `weeding_out_unusable_unapproved_drugs.ipynb` — direct precursor of v2's feature-completion/backfill logic (documented in `FINAL_DF_README.md`)

---

## Appendix — key artifact index

- Per-hypothesis raw data: `h{1,2,3,4}_*/adverse_*.parquet`, `non_interacting_*.parquet`
- Sparsified graphs: `h{1,2,3,4}_*/networks/*.graphml`
- node2vec embeddings: `h{1,2,3,4}_*/embeddings/combined_*_node2vec.parquet` (the only embeddings safe for downstream classifiers — never the per-population ones)
- Per-hypothesis RF comparison models: `h{1,2,3}_*/models/`
- Combined feature table (stale): `notebooks/unified_adverse_df.parquet`, `unified_non_interacting_df.parquet`
- Combined RF baseline (stale): `notebooks/models/combined_rf_model/`
- Ensemble model artifacts (current): `notebooks/models/ensemble_rf_dt_lr/`
- Short-form running results log: [RESULTS_09_4_26.md](RESULTS_09_4_26.md)
